# 02 — Compile formats

Same enforced graph, **different emitted shapes** for different consumers.

| Format | Purpose |
|--------|--------|
| `json` | Full graph artifact (includes compile warnings) |
| `catalog` | Catalog-oriented artifact (warnings stripped) |
| `openlineage` | OpenLineage interop |
| `text` | Human-readable summary |

Assumes you read **01** for raw → standardize context.

In [ ]:
import importlib.util
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from types import ModuleType

_COLD_START_GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/ClearMetric-Labs/ClearMetric-Core/main'

_CACHED_NOTEBOOKS_DIR = Path('/Users/jonkim/.cache/clearmetric/github-main/examples/notebooks')

def _github_raw_base() -> str:
    paths = sys.modules.get("_paths")
    default = (
        paths.GITHUB_RAW_BASE if paths is not None else _COLD_START_GITHUB_RAW_BASE
    )
    return os.environ.get("CM_CLEARMETRIC_GITHUB_RAW_BASE", default)
def _fetch_repo_file(repo_relative: str, dest: Path) -> None:
    if dest.is_file():
        return
    paths = sys.modules.get("_paths")
    if paths is not None:
        paths._fetch_github_file(repo_relative, dest)
        return
    url = f"{_github_raw_base()}/{repo_relative}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            dest.write_bytes(response.read())
    except urllib.error.URLError as exc:
        raise FileNotFoundError(
            f"Failed to download {url}. Check network access and branch path."
        ) from exc
def _find_local_helpers(start: Path | None = None) -> Path | None:
    start = start or Path.cwd()
    for root in (start, *start.parents):
        nested = root / "examples" / "notebooks"
        if (nested / "_paths.py").is_file():
            return nested
        if (root / "_paths.py").is_file() and (root / "_notebook_setup.py").is_file():
            return root
    return None
def _resolve_setup_file(start: Path | None = None) -> Path:
    local = _find_local_helpers(start)
    if local is not None:
        return local / "_notebook_setup.py"
    paths = sys.modules.get("_paths")
    cache_dir = (
        paths.CACHED_NOTEBOOKS_DIR if paths is not None else _CACHED_NOTEBOOKS_DIR
    )
    setup_file = cache_dir / "_notebook_setup.py"
    if not setup_file.is_file():
        _fetch_repo_file("examples/notebooks/_notebook_setup.py", setup_file)
    return setup_file
def _exec_setup_module(setup_file: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location("_notebook_setup", setup_file)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {setup_file}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
_exec_setup_module(_resolve_setup_file()).setup_notebook()


In [ ]:
import json

from _paths import lineage_demo_project

PROJECT_DIR = lineage_demo_project()


## Compile once

In [ ]:
from clearmetric.compiler import compile
from clearmetric.emitters import emit_compile
from clearmetric.emitters.registry import WEDGE_COMPILE_FORMATS

compiled = compile(PROJECT_DIR)
graph = json.loads(emit_compile("json", compiled))
catalog = json.loads(emit_compile("catalog", compiled))
ol = json.loads(emit_compile("openlineage", compiled))
text = emit_compile("text", compiled)

print("public wedge formats:", WEDGE_COMPILE_FORMATS)

## `json` — full graph

One real node and one real lineage edge.

In [ ]:
print(json.dumps({
    "top_level_keys": sorted(graph.keys()),
    "node_count": len(graph["nodes"]),
    "edge_count": len(graph["edges"]),
    "warning_count": len(graph["warnings"]),
}, indent=2))

example_node = next(n for n in graph["nodes"] if n["id"] == "column:orders_base.amount")
example_edge = next(e for e in graph["edges"] if e["source_id"] == "column:orders_base.amount")
print("\nexample node:")
print(json.dumps(example_node, indent=2))
print("\nexample edge:")
print(json.dumps(example_edge, indent=2))

## `catalog` — catalog contract

For this fixture, **the same physical nodes and edges** appear in `catalog`. The difference is the **emitted contract**: compile warnings are removed, and warehouse-bound columns include `bindings`.

In [ ]:
graph_node_ids = {n["id"] for n in graph["nodes"]}
catalog_node_ids = {n["id"] for n in catalog["nodes"]}
print(json.dumps({
    "catalog_node_count": len(catalog["nodes"]),
    "catalog_warning_count": len(catalog["warnings"]),
    "same_node_ids_as_graph": catalog_node_ids == graph_node_ids,
}, indent=2))

catalog_slice = [n for n in catalog["nodes"] if n["id"] in {
    "table:raw_orders", "column:raw_orders.amount", "column:orders_base.amount",
}]
print("\nexample catalog slice:")
print(json.dumps(catalog_slice, indent=2))

## `openlineage` — interop shape

Full job, datasets, and column lineage for this fixture.

In [ ]:
print(json.dumps({
    "top_level_keys": sorted(ol.keys()),
    "dataset_count": len(ol.get("datasets") or []),
    "column_lineage_count": len(ol.get("columnLineage") or []),
}, indent=2))
print("\nopenlineage payload:")
print(json.dumps({
    "job": ol["job"],
    "datasets": ol.get("datasets") or [],
    "columnLineage": ol.get("columnLineage") or [],
}, indent=2))

## `text` — CLI-style summary

In [ ]:
print(text)

## Validate

In [ ]:
from clearmetric.core.validate import validate_artifact_dict

validate_artifact_dict(graph)
validate_artifact_dict(catalog)
print("graph and catalog pass catalog-artifact.schema.json")